In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
print(sys.path[0])

/home/benjamin-rodriguez/Desktop/PredictionMarkets


In [4]:
import numpy as np
from analysis.correlation import (
    joint_table, frechet_bounds, is_consistent,
    implied_correlation, joint_from_correlation, correlation_range,
    conditionals, marginal_from_conditionals, reasonableness_report,
)

In [6]:
p_a = 0.35   # P(Messi over 3.5 shots)
p_b = 0.41   # P(Argentina wins)

lo, hi = frechet_bounds(p_a, p_b)
print(f"P(A&B) must lie in [{lo:.3f}, {hi:.3f}]")
print(f"independence would put it at p_a*p_b = {p_a*p_b:.3f}")

P(A&B) must lie in [0.000, 0.350]
independence would put it at p_a*p_b = 0.143


In [10]:
cmin, cmax = correlation_range(p_a, p_b)
print(f"feasible correlation range: [{cmin:+.3f}, {cmax:+.3f}]")

phi = 0.45   # your view: moderately positively correlated
p_ab = joint_from_correlation(p_a, p_b, phi)
print(f"phi={phi:+.2f}  ->  P(A&B)={p_ab:.3f}")

print("\njoint table:")
for k, v in joint_table(p_a, p_b, p_ab).items():
    print(f"  {k} = {v:.3f}")

feasible correlation range: [-0.612, +0.880]
phi=+0.45  ->  P(A&B)=0.249

joint table:
  p11 = 0.249
  p10 = 0.101
  p01 = 0.161
  p00 = 0.489


In [11]:
report = reasonableness_report(p_a, p_b, p_ab, expected_sign="positive")

marginals: P(A)=0.350  P(B)=0.410
joint:     P(A&B)=0.249   (independence would give 0.143)
Frechet bounds for P(A&B): [0.000, 0.350]  ->  CONSISTENT
joint table:  A&B=0.249   A&notB=0.101   notA&B=0.161   neither=0.489
correlation phi = +0.450   (feasible range [-0.612, +0.880])
conditionals: P(B|A)=0.712   P(B)=0.410   P(B|notA)=0.248
  monotonic: positive dependence (A makes B more likely)


In [13]:
# Suppose a third market prices P(A and B) directly, or you back out an implied joint that's too high:
bad_p_ab = 0.38     
print(f"claimed P(A&B)={bad_p_ab:.3f}, bounds {frechet_bounds(p_a, p_b)}")
print("consistent?", is_consistent(p_a, p_b, bad_p_ab))

claimed P(A&B)=0.380, bounds (0.0, 0.35)
consistent? False


In [ ]:
derived = marginal_from_conditionals(
    p_a=0.35,
    p_b_given_a=0.72,       
    p_b_given_not_a=0.43,   
)
q_b = derived["p_b"]
market_b = 0.55            # what B actually trades at
print(f"your fair price for B: q_B = {q_b:.3f}")
print(f"market price of B:     p_B = {market_b:.3f}")
print(f"edge (q_B - p_B) = {q_b - market_b:+.3f}")

your fair price for B: q_B = 0.532
market price of B:     p_B = 0.550
edge (q_B - p_B) = -0.018


Now a correlation range, so given your understanding of the market this is the plausible range for phi to live

In [8]:
phi_low, phi_high = 0.30, 0.60

lo_bound, hi_bound = frechet_bounds(p_a, p_b)
p_ab_lo = np.clip(joint_from_correlation(p_a, p_b, phi_low),  lo_bound, hi_bound)
p_ab_hi = np.clip(joint_from_correlation(p_a, p_b, phi_high), lo_bound, hi_bound)

print(f"correlation in [{phi_low:+.2f}, {phi_high:+.2f}] implies "
      f"P(A&B) in [{p_ab_lo:.3f}, {p_ab_hi:.3f}]")

correlation in [+0.30, +0.60] implies P(A&B) in [0.214, 0.284]


In [ ]:
def p_b_from_joint(p_a, p_ab):
    pba  = p_ab / p_a                       # P(B|A)
    pbna = (p_b - p_ab) / (1 - p_a)         # uses current p_b only to split
    return pba * p_a + pbna * (1 - p_a)


cond_base = conditionals(p_a, p_b, p_ab)        # p_ab from the base-case cell
pbna_fixed = cond_base["P(B|notA)"]

def fair_pb(p_a, p_ab, pbna):
    pba = p_ab / p_a
    return pba * p_a + pbna * (1 - p_a)

pb_lo = fair_pb(p_a, p_ab_lo, pbna_fixed)
pb_hi = fair_pb(p_a, p_ab_hi, pbna_fixed)
pb_lo, pb_hi = sorted((pb_lo, pb_hi))

market_b = 0.55
print(f"fair price for B across the correlation range: [{pb_lo:.3f}, {pb_hi:.3f}]")
print(f"B's market price: {market_b:.3f}")
if pb_lo <= market_b <= pb_hi:
    print("  market price is INSIDE your plausible range — no clear edge")
else:
    side = "underpriced (buy YES)" if market_b < pb_lo else "overpriced (buy NO)"
    print(f"  market price is OUTSIDE your range — B looks {side}")

fair price for B across the correlation range: [0.375, 0.445]
B's market price: 0.550
  market price is OUTSIDE your range — B looks overpriced (buy NO)
